In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/evaluator/evaluate.py
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone/lightgcn.py


In [3]:
import json
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import subprocess
import sys 

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")

sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator') 
from evaluate import evaluate

sys.path.append('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone') 
from lightgcn import LightGCN, build_lightgcn_adj_matrix, bpr_loss

# Bước 1: Nạp danh sách khóa lạnh từ các tệp JSON kịch bản cold
scenario_dir = "/kaggle/input/datasets/ngocccchannn/cold-scenario"
with open(f"{scenario_dir}/cold_items_val.json", "r", encoding="utf-8") as f:
    cold_val_ids = json.load(f)
with open(f"{scenario_dir}/cold_items_test.json", "r", encoding="utf-8") as f:
    cold_test_ids = json.load(f)

# Bước 3: Chuẩn bị target và historical mask cho tập test_cold
te_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet')
tr_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet')
va_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet')

# Xây dựng ánh xạ user2id (chỉ cần lấy từ tập train cold)
all_users = tr_cold['user_id'].unique()
user2id = {str(u): i for i, u in enumerate(all_users)}
num_users = len(user2id)

# Xây dựng ánh xạ item2id: Gộp toàn bộ item từ train_cold VÀ các item lạnh (val/test)
all_train_items = tr_cold['course_id'].unique()
all_cold_items = list(set(cold_test_ids) | set(cold_val_ids))
unique_all_items = list(set(all_train_items) | set(all_cold_items))

item2id = {str(item): i for i, item in enumerate(unique_all_items)}
num_items = len(item2id)

print(f"Tổng số User chính xác: {num_users}, Tổng số Item (đã bao gồm item lạnh): {num_items}")

train_users = torch.tensor([user2id[str(u)] for u in tr_cold['user_id']], dtype=torch.long)
train_items = torch.tensor([item2id[str(i)] for i in tr_cold['course_id']], dtype=torch.long)

# Xây dựng ma trận kề chuẩn hóa từ toàn bộ train_cold
print("Đang xây dựng ma trận kề đồ thị...")
adj_mat = build_lightgcn_adj_matrix(train_users, train_items, num_users, num_items, DEVICE)

Sử dụng thiết bị: cuda
Tổng số User chính xác: 98220, Tổng số Item (đã bao gồm item lạnh): 3154
Đang xây dựng ma trận kề đồ thị...


/kaggle/input/datasets/ngocccchannn/lightgcn-backbone/lightgcn.py:46: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (total_nodes, total_nodes)).to(device)


In [4]:
# --- 4. HUẤN LUYỆN MÔ HÌNH LIGHTGCN MỚI ---
LATENT_DIM = 64
N_LAYERS = 3
BATCH_SIZE = 1024
LR = 1e-3
EPOCHS = 40

model = LightGCN(num_users, num_items, latent_dim=LATENT_DIM, n_layers=N_LAYERS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

print("\n--- Bắt đầu huấn luyện LightGCN mới trên toàn bộ train_cold ---")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    perm = torch.randperm(len(train_users))
    sampled_u = train_users[perm]
    sampled_pos_i = train_items[perm]
    sampled_neg_i = torch.randint(0, num_items, (len(perm),), dtype=torch.long)
    
    for s in range(0, len(perm), BATCH_SIZE):
        batch_u = sampled_u[s:s+BATCH_SIZE].to(DEVICE)
        batch_pos = sampled_pos_i[s:s+BATCH_SIZE].to(DEVICE)
        batch_neg = sampled_neg_i[s:s+BATCH_SIZE].to(DEVICE)
        
        optimizer.zero_grad()
        u_emb, i_emb = model(adj_mat)
        loss = bpr_loss(batch_u, batch_pos, batch_neg, u_emb, i_emb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f}")

print("Huấn luyện hoàn thành!")

# Lưu embedding mới ra file
model.eval()
with torch.no_grad():
    u_emb_final, i_emb_final = model(adj_mat)
    np.save('g_u.npy', u_emb_final.cpu().numpy())
    np.save('g_i.npy', i_emb_final.cpu().numpy())
print("Đã lưu thành công file embedding mới: 'g_u.npy' và 'g_i.npy'.")


--- Bắt đầu huấn luyện LightGCN mới trên toàn bộ train_cold ---
Epoch 1/40 | Loss: 133.3548
Epoch 5/40 | Loss: 58.9786
Epoch 10/40 | Loss: 42.0060
Epoch 15/40 | Loss: 33.9210
Epoch 20/40 | Loss: 28.5804
Epoch 25/40 | Loss: 25.1710
Epoch 30/40 | Loss: 23.2948
Epoch 35/40 | Loss: 21.5256
Epoch 40/40 | Loss: 20.2592
Huấn luyện hoàn thành!
Đã lưu thành công file embedding mới: 'g_u.npy' và 'g_i.npy'.


In [8]:
# Hàm tính điểm (scoring function) dùng ma trận nhân chấm
def score_fn_cold(users_tensor):
    u_idx = users_tensor.to(DEVICE)
    return torch.matmul(u_emb_final[u_idx], i_emb_final.t())

# Chuẩn bị target_cold và hist_cold cho tập test
target_cold = {}
for u, group in te_cold.groupby('user_id'):
    u_str = str(u)
    if u_str in user2id:
        u_idx = user2id[u_str]
        item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
        if len(item_idxs) > 0:
            target_cold[u_idx] = item_idxs

hist_cold_df = pd.concat([tr_cold, va_cold])
hist_cold = {}
for u, group in hist_cold_df.groupby('user_id'):
    u_str = str(u)
    if u_str in user2id:
        u_idx = user2id[u_str]
        item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
        if len(item_idxs) > 0:
            hist_cold[u_idx] = item_idxs

In [9]:
# --- Kịch bản (a): Chỉ tập trung đánh giá trên các khóa lạnh (Cold Items Candidate) ---
cand_cold_indices = [item2id[str(c)] for c in cold_test_ids if str(c) in item2id]
cand_bool_mask_a = np.zeros(num_items, dtype=bool)
if len(cand_cold_indices) > 0:
    cand_bool_mask_a[cand_cold_indices] = True

print("\n--- Đang đánh giá LightGCN trên Kịch bản Cold (a): Chỉ ứng viên lạnh ---")
res_cold_a = evaluate(
    score_fn_cold, 
    target_cold, 
    hist_cold, 
    ks=(5, 10, 20), 
    cand=cand_bool_mask_a, 
    bs=1024
)
for k, v in res_cold_a.items():
    print(f"{k}: {v.mean():.4f}")


--- Đang đánh giá LightGCN trên Kịch bản Cold (a): Chỉ ứng viên lạnh ---
Recall@5: 0.0380
Recall@10: 0.0798
Recall@20: 0.0926
NDCG@5: 0.0212
NDCG@10: 0.0373
NDCG@20: 0.0411
HR@5: 0.0674
HR@10: 0.1377
HR@20: 0.1554


In [10]:
# --- Kịch bản (b): Toàn bộ không gian khóa học (All Items Candidate) ---
cand_bool_mask_b = np.ones(num_items, dtype=bool)

print("\n--- Đang đánh giá LightGCN trên Kịch bản Cold (b): Toàn bộ item ---")
res_cold_b = evaluate(
    score_fn_cold, 
    target_cold, 
    hist_cold, 
    ks=(5, 10, 20), 
    cand=cand_bool_mask_b, 
    bs=1024
)
for k, v in res_cold_b.items():
    print(f"{k}: {v.mean():.4f}")


--- Đang đánh giá LightGCN trên Kịch bản Cold (b): Toàn bộ item ---
Recall@5: 0.0000
Recall@10: 0.0000
Recall@20: 0.0000
NDCG@5: 0.0000
NDCG@10: 0.0000
NDCG@20: 0.0000
HR@5: 0.0000
HR@10: 0.0000
HR@20: 0.0000
